# Banking Model Representation – Extended Solution

**Domain:** Banking / Credit Risk / Retail Lending (Income-based underwriting)  
**Inspired by:** Coursera Machine Learning Specialization – C1_W1 Lab02 Model Representation  
**Extended with:** vectorized alternate, residual analysis, more practice, parameterised simulation, audience notes, flowchart

![Flowchart of the workflow](banking_model_representation_flowchart.png)


## Goals
In this lab you will:
- Implement the model \( f_{w,b} \) for linear regression with one variable
- Apply it to a **banking** underwriting problem (applicant income → maximum loan amount)
- Explore alternate implementations, residuals, and a simulation that lets you modify credit-policy parameters


## Notation (Banking context)
| Notation | Description | Python |
|:---------|:------------|:-------|
| \( a \) | scalar | |
| \( \mathbf{a} \) | vector | |
| \( \mathbf{x} \) | Applicant Income (units of $10,000) | `x_train` |
| \( \mathbf{y} \) | Max Loan Amount Approved (units of $10,000) | `y_train` |
| \( x^{(i)}, y^{(i)} \) | \( i \)-th applicant | `x_i`, `y_i` |
| \( m \) | Number of observed credit decisions | `m` |
| \( w \) | slope / income multiplier (marginal loan per income unit) | `w` |
| \( b \) | intercept / base loan limit | `b` |
| \( f_{w,b}(x^{(i)}) \) | Predicted max loan = \( w x^{(i)} + b \) | `f_wb` |


## 0. Imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except Exception:
    pass

print("Libraries imported successfully.")


## 1. Problem Statement – Banking / Credit Underwriting

A retail bank has data from two recent personal-loan decisions:

| Applicant Annual Income (units of $10,000) | Maximum Loan Amount Approved (units of $10,000) |
|--------------------------------------------|-------------------------------------------------|
| 1.0                                        | 300                                             |
| 2.0                                        | 500                                             |

**Banking interpretation**
- \( x \): applicant’s annual income (scaled)
- \( y \): maximum loan amount the bank is willing to approve
- The linear model \( f_{w,b}(x) = wx + b \) is the simplest income-based underwriting rule used in retail credit policy.

We will fit a line through these two decisions and then forecast the loan limit for a new applicant with income = 1.2 units ($12,000).


### Create training data


In [ ]:
# x_train is the input variable (Applicant Income in units of $10k)
# y_train is the target (Max Loan Amount in units of $10k)
x_train = np.array([1.0, 2.0])
y_train = np.array([300.0, 500.0])
print(f"x_train = {x_train}")
print(f"y_train = {y_train}")


### Number of training examples `m`


In [ ]:
print(f"x_train.shape: {x_train.shape}")
m = x_train.shape[0]
print(f"Number of training examples is: {m}")

# Alternative
m = len(x_train)
print(f"Number of training examples (via len): {m}")


### Training example \( x^{(i)}, y^{(i)} \)


In [ ]:
i = 0  # Change this to 1 to see (x^(1), y^(1))
x_i = x_train[i]
y_i = y_train[i]
print(f"(x^({i}), y^({i})) = ({x_i}, {y_i})")


## 2. Plotting the Banking Data


In [ ]:
# Plot the two credit decisions
plt.figure(figsize=(7, 5))
plt.scatter(x_train, y_train, marker='x', c='r', s=100, label='Actual Credit Decisions')
plt.title("Applicant Income vs Maximum Loan Amount")
plt.ylabel('Max Loan Amount (units of $10,000)')
plt.xlabel('Applicant Income (units of $10,000)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 3. Model Function

\[ f_{w,b}(x^{(i)}) = w x^{(i)} + b \]

Start with an arbitrary guess \( w = 100 \), \( b = 100 \).


In [ ]:
w = 100
b = 100
print(f"w: {w}")
print(f"b: {b}")


### Loop-based implementation of `compute_model_output`


In [ ]:
def compute_model_output(x, w, b):
    """
    Computes the prediction of a linear model (explicit loop)
    Args:
      x (ndarray (m,)): Data, m examples 
      w, b (scalar)   : model parameters  
    Returns
      f_wb (ndarray (m,)): model prediction
    """
    m = x.shape[0]
    f_wb = np.zeros(m)
    for i in range(m):
        f_wb[i] = w * x[i] + b
    return f_wb


### Plot prediction vs actual (initial parameters)


In [ ]:
tmp_f_wb = compute_model_output(x_train, w, b)

plt.figure(figsize=(7, 5))
plt.plot(x_train, tmp_f_wb, c='b', label='Our Prediction (w=100, b=100)')
plt.scatter(x_train, y_train, marker='x', c='r', s=100, label='Actual Credit Decisions')
plt.title("Applicant Income vs Maximum Loan Amount")
plt.ylabel('Max Loan Amount (units of $10,000)')
plt.xlabel('Applicant Income (units of $10,000)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("With w=100, b=100 the line does NOT fit the observed credit decisions.")


### Challenge – Fit the data
The unique line that passes through both points has slope

\[ w = \frac{500-300}{2-1} = 200 \]

and intercept \( b = 100 \).


In [ ]:
w = 200
b = 100
print(f"Fitted parameters → w: {w}, b: {b}")

tmp_f_wb = compute_model_output(x_train, w, b)

plt.figure(figsize=(7, 5))
plt.plot(x_train, tmp_f_wb, c='b', linewidth=2, label='Fitted Model (w=200, b=100)')
plt.scatter(x_train, y_train, marker='x', c='r', s=100, label='Actual Credit Decisions')
plt.title("Fitted Income-Based Underwriting Rule")
plt.ylabel('Max Loan Amount (units of $10,000)')
plt.xlabel('Applicant Income (units of $10,000)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 4. Prediction for a New Applicant
Forecast the maximum loan amount for an applicant with annual income = 1.2 units ($12,000).


In [ ]:
w = 200
b = 100
x_i = 1.2
predicted_loan = w * x_i + b

print(f"Predicted max loan amount for $12,000 income: {predicted_loan:.0f} units")
print(f"→ approximately ${predicted_loan * 10_000:,.0f} (if each unit = $10,000)")


## 5. Alternate Implementation – Fully Vectorized
No Python `for` loop; NumPy broadcasting does the arithmetic in one expression.


In [ ]:
def compute_model_output_vectorized(x, w, b):
    """Vectorized linear model: f = w * x + b"""
    return w * x + b

# Verify equivalence
print("Loop version :", compute_model_output(x_train, 200, 100))
print("Vector version:", compute_model_output_vectorized(x_train, 200, 100))
print("Identical?", np.allclose(compute_model_output(x_train, 200, 100),
                               compute_model_output_vectorized(x_train, 200, 100)))


## 6. More Practice Exercises


In [ ]:
# Practice 1 – Add a third synthetic applicant and re-plot
x_ext = np.array([1.0, 1.5, 2.0])
y_ext = np.array([300.0, 400.0, 500.0])   # still perfectly linear

plt.figure(figsize=(7, 5))
plt.scatter(x_ext, y_ext, marker='x', c='r', s=100, label='Applicants (incl. synthetic)')
plt.plot(x_ext, 200 * x_ext + 100, 'b-', label='Same fitted underwriting rule')
plt.title("Extended Data Still Lies on the Same Line")
plt.xlabel('Applicant Income (units $10k)')
plt.ylabel('Max Loan Amount (units $10k)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Practice 2 – Predictions for additional incomes
for income in [0.8, 1.2, 2.5]:
    pred = 200 * income + 100
    print(f"Income {income:4.1f} → predicted max loan {pred:6.1f} units")

# Practice 3 – Residuals on original data (should be ~0 with perfect fit)
residuals = y_train - compute_model_output_vectorized(x_train, 200, 100)
print("\nResiduals (actual - predicted):", residuals)
print("Sum of squared residuals:", np.sum(residuals**2))


## 7. Simulation Section – Modify Parameters & Observe Results

Change any of the values in the parameter block and re-run the cell.  
You will see:
- How the predicted loan limit for a chosen income changes
- A family of underwriting lines for different income multipliers (policy scenarios)
- Sensitivity of the loan offer to the base limit (b)


In [ ]:
# ========== SIMULATION PARAMETERS (edit freely) ==========
sim_w     = 200.0      # slope (income multiplier / credit policy lever)
sim_b     = 100.0      # intercept (base loan limit)
x_new     = 1.2        # new applicant income to forecast
n_lines   = 5          # how many alternative slopes to draw
w_spread  = 50.0       # ± range around sim_w for the family of lines
# ========================================================

# Single forecast
forecast = sim_w * x_new + sim_b
print(f"Forecast for income={x_new}: {forecast:.1f} loan units")
print(f"  (w={sim_w}, b={sim_b})")

# Visual sensitivity: family of underwriting lines
plt.figure(figsize=(8, 5))
x_grid = np.linspace(0.5, 2.5, 50)
for dw in np.linspace(-w_spread, w_spread, n_lines):
    w_alt = sim_w + dw
    plt.plot(x_grid, w_alt * x_grid + sim_b, alpha=0.6,
             label=f"w={w_alt:.0f}")
plt.scatter(x_train, y_train, marker='x', c='r', s=120, zorder=5, label='Observed Decisions')
plt.axvline(x_new, color='gray', ls='--', alpha=0.7)
plt.scatter([x_new], [forecast], c='green', s=120, zorder=6, label=f'Forecast @ {x_new}')
plt.title("Sensitivity of Loan Limit to Income Multiplier (w)")
plt.xlabel('Applicant Income (units of $10,000)')
plt.ylabel('Max Loan Amount (units of $10,000)')
plt.legend(loc='upper left', fontsize=8)
plt.grid(True, alpha=0.3)
plt.show()

# Quick numeric sensitivity table
print("\nSensitivity table (change in loan limit when w or b changes by ±10%):")
base = sim_w * x_new + sim_b
for label, w_, b_ in [("base", sim_w, sim_b),
                      ("w +10%", sim_w*1.1, sim_b),
                      ("w -10%", sim_w*0.9, sim_b),
                      ("b +10%", sim_w, sim_b*1.1),
                      ("b -10%", sim_w, sim_b*0.9)]:
    print(f"  {label:8s}: loan limit = {w_*x_new + b_:.1f}  (Δ = {w_*x_new + b_ - base:+.1f})")


## 8. Audience Adaptation Notes (Banking stakeholders)

**Primary audience:** Credit risk analysts and underwriters who already understand scatterplots, slope, and intercept.  
→ Keep the mathematical notation, show residuals, allow them to change policy parameters (w, b).

**Secondary audience (Credit Committee / executives):**  
→ Emphasize the single headline number (“For a $12k-income applicant we would approve a max loan of ~$3.4 M under current policy”) and the visual of the fitted underwriting line.  
→ Avoid talking about “vectorized implementations” or “sum of squared residuals” unless asked.

**Data-literacy tip:** Highly data-literate credit officers are comfortable with the model equation; relationship managers and senior executives prefer the plain-English story + one clean chart + sensitivity band.


## Congratulations!
In this extended lab you have learned:
- Linear regression builds a model that relates a banking feature (applicant income) to a target (max loan amount)
- The model has two parameters \( w \) (income multiplier) and \( b \) (base limit) that are “fit” to observed credit decisions
- Once parameters are known, the model produces loan-limit forecasts for novel applicants
- Vectorized NumPy code is concise and fast; loops are pedagogically clear
- A small simulation reveals how sensitive the loan offer is to the chosen income multiplier and base limit — useful for credit-policy discussions

These ideas are the foundation for cost functions, gradient descent, and ultimately automated or scorecard-based learning of \( w \) and \( b \) under regulatory constraints.
